### Transform Results Data (Bronze to Silver)

**Steps:**
1. Read the raw data from the Bronze table
2. Drop the `url` column
3. Rename columns to snake_case and meaningful names
4. Remove rows with missing key values (season, round, constructor_id, driver_id)
5. Remove duplicates based on driver_id, season, constructor_id, round
6. Apply title case to `race_name`
7. Save to Silver table

### Loading Configuration and Setting Variables

In [0]:
%run ../00-common/01.environment-config 

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.results'
silver_table = f'{catalog_name}.{silver_schema}.results'

In [0]:
from pyspark.sql.functions import *

### Step 1: Read Bronze Table
Read the raw results data from `formula1.bronze.results` into a DataFrame.

In [0]:
# Read Bronze table
results_df = spark.read.table(bronze_table)
display(results_df)

### Step 2: Drop Columns
Remove the `url` column - not needed for analysis.

In [0]:
# Drop unnecessary columns
results_drop_df = results_df.drop('url')
display(results_drop_df)

### Step 3: Rename Columns
Rename columns to snake_case for consistency:
- `constructorId` > `constructor_id`
- `driverId` > `driver_id`
- `raceName` > `race_name`
- `date` > `day`
- `grid` > `grid_position`
- `laps` > `Completed_laps`
- `number` > `car_number`
- `position` > `final_position`
- `positionText` > `final_position_text`

In [0]:
# Rename columns to snake_case
results_renamed_df = (results_drop_df
    .withColumnsRenamed({'constructorId': 'constructor_id',
                         'driverId': 'driver_id',
                         'raceName': 'race_name',
                         'date': 'race_date',
                         'grid' : 'grid_position',
                         'laps':'Completed_laps',
                         'number' : 'car_number',
                         'position':'final_position',
                         'positionText':'final_position_text'})
)
display(results_renamed_df)

### Step 4: Remove Null Values
Filter out rows where key columns are null. If `season`, `round`, `constructor_id`, or `driver_id` is missing, the row is useless.

In [0]:
# Remove rows with null key values
results_valid_df = (
    results_renamed_df
    .filter(
        col('season').isNotNull() &
        col('round').isNotNull() &
        col('constructor_id').isNotNull() &
        col('driver_id').isNotNull()
    )
)
display(results_valid_df)

### Step 5: Remove Duplicates
Drop duplicate rows based on `driver_id`, `season`, `constructor_id`, and `round`. Each driver should have only one result per race.

In [0]:
# Remove duplicates
results_distinct_df = results_valid_df.dropDuplicates(['driver_id','season','constructor_id','round'])
display(results_distinct_df)

### Step 6: Title Case
Convert `race_name` to title case using `initcap()` (e.g., "british grand prix" becomes "British Grand Prix").

In [0]:
# Apply title case to race_name
results_final_df = results_distinct_df.withColumn('race_name', initcap(col('race_name')))
display(results_final_df)

### Step 7: Write to Silver Table
Save the final cleaned DataFrame to `formula1.silver.results` in Delta format with overwrite mode.

In [0]:
(
    results_final_df.write
    .mode('overwrite')
    .option('overwriteSchema', True)
    .format('delta')
    .saveAsTable(silver_table)
)

print(f'Written to {silver_table} successfully')

### Verify
Read back the Silver table to confirm data was written correctly.

In [0]:
# Verify
spark.read.table(silver_table).display()